# Data Cleaning — Parkinson's Disease (Longitudinal + Single-session)
Load the raw parquet tables saved from `explore_longitudinal.ipynb`, inspect, clean, and prepare for modelling.

In [211]:
import numpy as np
import pandas as pd
from pathlib import Path

# ── Load the two Parkinson's tables ──────────────────────────────
DATA_DIR = Path("/home/srl/B2AI/LLM/data")

pk_train = pd.read_parquet(DATA_DIR / "raw_data_non_longitudinal_parkinson.parquet")

## 1. Inspect raw data
Show column types, NaN percentages, and a few sample rows for each table.

In [212]:
def inspect_table(df, name):
    """Print detailed column-level summary for a DataFrame."""
    print(f"\n{'═' * 80}")
    print(f"  {name}   shape: {df.shape}")
    print(f"{'═' * 80}")

    # Columns that contain arrays (not scalar) — skip nunique/sample for these
    ARRAY_COLS = {"mfcc", "mel_spectrogram"}

    rows = len(df)
    report = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        n_na = df[col].isna().sum()
        pct_na = 100 * n_na / rows if rows > 0 else 0

        if col in ARRAY_COLS:
            n_unique = "—"
            sample = f"<array len={len(df[col].dropna().iloc[0]) if n_na < rows else 0}>"
        else:
            n_unique = df[col].nunique()
            non_null = df[col].dropna()
            sample = str(non_null.iloc[0])[:40] if len(non_null) > 0 else "—"

        report.append({
            "column": col, "dtype": dtype,
            "NaN": n_na, "NaN%": round(pct_na, 1),
            "unique": n_unique, "sample": sample,
        })

    report_df = pd.DataFrame(report)
    display(report_df)

    # Highlight columns with >60% NaN
    high_nan = report_df[report_df["NaN%"] > 60]
    if len(high_nan) > 0:
        print(f"\n  ⚠  {len(high_nan)} columns with > 60% NaN:")
        for _, r in high_nan.iterrows():
            print(f"    • {r['column']:55s}  {r['NaN%']:5.1f}%")
    else:
        print(f"\n  ✔  No columns with > 60% NaN")

    return report_df

report = inspect_table(pk_train, "pk_train (non-longitudinal)")


════════════════════════════════════════════════════════════════════════════════
  pk_train (non-longitudinal)   shape: (1876, 135)
════════════════════════════════════════════════════════════════════════════════


,column,dtype,NaN,NaN%,unique,sample
0,participant_id,str,0,0.0,56,110043
1,session_id,str,0,0.0,56,2cd1e0d6
2,task_name,str,0,0.0,120,cinderella-story
3,mfcc,object,0,0.0,—,<array len=60>
4,n_frames_mfcc,int64,0,0.0,985,3278
...,...,...,...,...,...,...
130,eligible_studies___4,str,1732,92.3,1,1.0
131,eligible_studies___age_10_plus,str,1876,100.0,0,—
132,eligible_studies___age_2_4,str,1876,100.0,0,—
133,eligible_studies___age_4_6,str,1876,100.0,0,—



  ⚠  61 columns with > 60% NaN:
    • colorblind                                                91.0%
    • live_by_yourself                                          98.2%
    • primary_transportation                                    85.4%
    • roommate                                                 100.0%
    • sex_assigned_at_birth                                     91.0%
    • tone_deaf                                                 91.0%
    • transportation                                           100.0%
    • demographics_completed_by___2                             92.3%
    • demographics_completed_by___3                             98.0%
    • employ_status___1                                         98.1%
    • employ_status___10                                       100.0%
    • employ_status___11                                       100.0%
    • employ_status___12                                       100.0%
    • employ_status___13                                 

## 2. Drop columns with > 60% NaN
Remove columns that are mostly empty — they carry little information and add noise.

In [213]:
NAN_THRESHOLD = 0.60  # drop columns with more than 60% NaN

def drop_high_nan(df, name, threshold=NAN_THRESHOLD):
    """Drop columns with NaN% > threshold. Returns cleaned DataFrame."""
    n_rows = len(df)
    nan_pct = df.isna().sum() / n_rows
    drop_cols = nan_pct[nan_pct > threshold].index.tolist()

    print(f"{'═' * 70}")
    print(f"  {name}")
    print(f"  Before: {df.shape[1]} columns")
    print(f"  Dropping {len(drop_cols)} columns with > {threshold*100:.0f}% NaN:")

    df_clean = df.drop(columns=drop_cols)
    print(f"  After:  {df_clean.shape[1]} columns")
    print(f"{'═' * 70}\n")
    return df_clean

pk_train_c = drop_high_nan(pk_train, "pk_train (non-longitudinal)")

══════════════════════════════════════════════════════════════════════
  pk_train (non-longitudinal)
  Before: 135 columns
  Dropping 61 columns with > 60% NaN:
  After:  74 columns
══════════════════════════════════════════════════════════════════════



## 3. Column mappings (from `demographics.json`)

Five kinds of encoding, derived from the data dictionary:

| Kind | Columns | Rule |
|------|---------|------|
| **Ordinal** | `edu_level`, `household_income_usa`, `housing_status` | Text label → increasing integer. **"Prefer not to answer"** / **"Other"** → `NaN`. |
| **Boolean** | `cognition`, `hearing`, `mobility`, `sex_at_birth`, … | `Yes → 1`, `No → 0`, `Prefer not to answer → NaN`. |
| **Checked** | `self_reported_*` | `Checked → 1`, `Unchecked → 0`. |
| **One-hot** | `country`, `sexual_orientation` | Dummy columns per category. `Prefer not to answer → NaN`. |

In [214]:
# ══════════════════════════════════════════════════════════════════
#  3a. ORDINAL columns  — text → increasing integer
#      "Prefer not to answer" / "Other" are EXCLUDED → become NaN
# ══════════════════════════════════════════════════════════════════

ORDINAL_MAP = {
    # ── Education level (1 = lowest → 10 = highest) ─────────────
    "edu_level": {
        "No formal education":                              1,
        "Some elementary school":                           2,
        "Some secondary or high school education":          3,
        "High School or secondary school degree complete":  4,
        "Some college education":                           5,
        "Associate's or technical degree complete":         6,
        "College or baccalaureate degree complete":         7,
        "Some post-baccalaureate education":                8,
        "Graduate or professional degree complete":         9,
        "Doctoral or post graduate education":             10,
        # "Other" (11 in JSON) → NaN  — excluded from map
        # "Prefer not to answer" (12 in JSON) → NaN  — excluded from map
    },

    # ── Household income  (1 = lowest → 8 = highest) ────────────
    "household_income_usa": {
        "< $15,000":              1,
        "$15,000 to $29,999":     2,
        "$30,000 to $$49,999":    3,   # double-$ typo in source JSON
        "$30,000 to $49,999":     3,   # in case data has corrected form
        "$50,000 to $99,999":     4,
        "$100,000 to $149,999":   5,
        "$150,000 to $199,999":   6,
        "$200,000 to $249,999":   7,
        ">$250,000":              8,
        # "Prefer not to answer" (9 in JSON) → NaN  — excluded from map
    },

    # ── Housing status (1 = most dependent → 3 = most independent) ──
    "housing_status": {
        "Assisted living":                          1,
        "Skilled nursing facility/nursing home":    1,   # same level as assisted
        "Rent home":                                2,
        "Own home":                                 3,
        # Any other / unknown → NaN
    },
}

In [215]:
# ══════════════════════════════════════════════════════════════════
#  3b. BOOLEAN columns  — Yes→1, No→0, "Prefer not to answer"→NaN
#      Kept as a separate list so it's easy to iterate
# ══════════════════════════════════════════════════════════════════

BOOL_COLS = [
    # Demographics — disability / function questions
    "cognition",            # difficulty concentrating / remembering
    "hearing",              # deaf or serious difficulty hearing
    "independent_living",   # difficulty doing errands alone
    "mobility",             # difficulty walking / climbing stairs
    "self_care",            # difficulty dressing / bathing
    "vision",               # blind or serious difficulty seeing
    # Demographics — household composition (JSON: Yes=1, No=0)
    "children",
    "grandparent",
    "parent",
    "other_live_with",
    "spouse_partner_sig_other",
    # Ethnicity (Hispanic or Latino = yes / Not Hispanic or Latino = no)
    "ethnicity",
    # Enrollment flags
    "is_control_participant",
    "ef_any_cognitive_challeges",
    "ef_any_hearing_related_req",
    "ef_any_physical_challenges",
    "ef_any_vision_related_req",
    "sex_at_birth",
]

# Master mapping — covers all text variants found in the data
BOOL_MAP = {
    # Standard yes / no
    "Yes": 1,   "yes": 1,
    "No":  0,   "no":  0,
    # Ethnicity-specific labels
    "Hispanic or Latino":       1,
    "Not Hispanic or Latino":   0,
    # All ambiguous / refusal answers → NaN
    "Prefer not to answer":  np.nan,
    "noAnswer":              np.nan,
    "Not certain":           np.nan,
    "notCertain":            np.nan,
    "Male":                 1,
    "Female":               0,
}

# ══════════════════════════════════════════════════════════════════
#  3c. CHECKED columns  — Checked→1, Unchecked→0
#      (self_reported_* from enrollment)
# ══════════════════════════════════════════════════════════════════

CHECKED_MAP = {"Checked": 1, "Unchecked": 0}

In [216]:
# ══════════════════════════════════════════════════════════════════
#  3d. Apply all mappings + coerce remaining numeric columns
# ══════════════════════════════════════════════════════════════════

# Columns that should be coerced to numeric (already numbers or convertible)
NUMERIC_COLS = [
    "parkinsons_disease__diagnosis_parkinsons_ds",
    "diagnosis_parkinsons_ds",
    "vhi10__vhi_10_calc_score",
    "voice_perception__voice_quality_perception",
    "age",
    "household_count",
]

# Columns to drop (form-fill time / single-value — not useful for modelling)
DROP_COLS = ["demographics_duration", "ef_primary_language", "ef_select_language"]

# ── One-hot encoding targets ────────────────────────────────────
#  country:            USA / Canada  → one-hot (drop_first → 1 binary col)
#  sexual_orientation: Heterosexual / Bisexual  (Prefer not to answer → NaN first)
ONE_HOT_COLS = ["country", "sexual_orientation"]

# Values that should become NaN before one-hot encoding
ONE_HOT_NAN_VALUES = {"Prefer not to answer", "noAnswer"}


def clean_types(df, name):
    """Apply all mappings, coerce types, and drop useless columns."""
    df = df.copy()

    # ── 1. Drop useless columns ──────────────────────────────────
    existing_drop = [c for c in DROP_COLS if c in df.columns]
    if existing_drop:
        df = df.drop(columns=existing_drop)
        print(f"  [{name}] Dropped: {existing_drop}")

    # ── 2. Ordinal mapping ───────────────────────────────────────
    n_ordinal = 0
    for col, mapping in ORDINAL_MAP.items():
        if col not in df.columns:
            continue
        before_na = df[col].isna().sum()
        df[col] = df[col].map(mapping)          # unmapped values → NaN
        after_na = df[col].isna().sum()
        new_na = after_na - before_na
        n_ordinal += 1
        print(f"  [{name}] Ordinal  {col:40s}  ({new_na} unmapped → NaN)")

    # ── 3. Boolean mapping ───────────────────────────────────────
    n_bool = 0
    for col in BOOL_COLS:
        if col not in df.columns:
            continue
        before_na = df[col].isna().sum()
        df[col] = df[col].map(BOOL_MAP)         # unmapped values → NaN
        after_na = df[col].isna().sum()
        new_na = after_na - before_na
        n_bool += 1
        if new_na > 0:
            print(f"  [{name}] Bool     {col:40s}  ({new_na} → NaN)")

    # ── 4. Checked / Unchecked mapping (self_reported_*) ─────────
    checked_cols = [c for c in df.columns if c.startswith("self_reported_")]
    for col in checked_cols:
        df[col] = df[col].map(CHECKED_MAP)

    # ── 5. One-hot encoding (country, sexual_orientation) ────────
    n_onehot = 0
    for col in ONE_HOT_COLS:
        if col not in df.columns:
            continue
        # "Prefer not to answer" → NaN before encoding
        df[col] = df[col].where(~df[col].isin(ONE_HOT_NAN_VALUES))
        dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False)
        dummies = dummies.astype("Int8")         # nullable int to preserve NaN rows
        # Where original was NaN → set all dummies to NaN
        nan_mask = df[col].isna()
        dummies[nan_mask] = pd.NA
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)
        n_onehot += 1
        print(f"  [{name}] One-hot  {col:40s}  → {list(dummies.columns)}")

    # ── 6. Coerce remaining numeric columns (age, scores, etc.) ──
    coerced = []
    for col in NUMERIC_COLS:
        if col in df.columns:
            before_na = df[col].isna().sum()
            df[col] = pd.to_numeric(df[col], errors="coerce")
            after_na = df[col].isna().sum()
            new_na = after_na - before_na
            coerced.append(col)
            if new_na > 0:
                print(f"  [{name}] Numeric  {col}: {new_na} non-numeric → NaN")

    # ── 7. Coerce flag columns (___) to numeric ──────────────────
    flag_cols = [c for c in df.columns if "___" in c]
    for col in flag_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── Summary ──────────────────────────────────────────────────
    print(f"\n  [{name}] SUMMARY:")
    print(f"    Ordinal mapped:   {n_ordinal} columns")
    print(f"    Boolean mapped:   {n_bool} columns")
    print(f"    Checked mapped:   {len(checked_cols)} self_reported_* columns")
    print(f"    One-hot encoded:  {n_onehot} columns")
    print(f"    Numeric coerced:  {len(coerced)} columns")
    print(f"    Flag coerced:     {len(flag_cols)} (___) columns")
    print(f"    Final shape:      {df.shape}")

    return df


pk_train_c = clean_types(pk_train_c, "pk_train_c")

  [pk_train_c] Dropped: ['demographics_duration', 'ef_primary_language', 'ef_select_language']
  [pk_train_c] Ordinal  edu_level                                 (102 unmapped → NaN)
  [pk_train_c] Ordinal  household_income_usa                      (170 unmapped → NaN)
  [pk_train_c] Ordinal  housing_status                            (0 unmapped → NaN)
  [pk_train_c] Bool     cognition                                 (34 → NaN)
  [pk_train_c] Bool     hearing                                   (36 → NaN)
  [pk_train_c] Bool     independent_living                        (34 → NaN)
  [pk_train_c] Bool     ethnicity                                 (35 → NaN)
  [pk_train_c] One-hot  country                                   → ['country_Canada', 'country_USA']
  [pk_train_c] One-hot  sexual_orientation                        → ['sexual_orientation_Bisexual', 'sexual_orientation_Heterosexual (straight)']

  [pk_train_c] SUMMARY:
    Ordinal mapped:   3 columns
    Boolean mapped:   18 columns


In [217]:
list(pk_train_c["task_name"].unique())



['cinderella-story',
 'diadochokinesis-buttercup',
 'diadochokinesis-ka',
 'diadochokinesis-pa',
 'diadochokinesis-pataka',
 'diadochokinesis-ta',
 'glides-high-to-low',
 'glides-low-to-high',
 'loudness',
 'maximum-phonation-time-1',
 'maximum-phonation-time-2',
 'maximum-phonation-time-3',
 'picture-description',
 'productive-vocabulary-1',
 'productive-vocabulary-2',
 'productive-vocabulary-3',
 'productive-vocabulary-4',
 'productive-vocabulary-5',
 'productive-vocabulary-6',
 'prolonged-vowel',
 'rainbow-passage',
 'random-item-generation',
 'respiration-and-cough-breath-1',
 'respiration-and-cough-breath-2',
 'respiration-and-cough-cough-1',
 'respiration-and-cough-cough-2',
 'respiration-and-cough-fivebreaths-1',
 'respiration-and-cough-fivebreaths-2',
 'respiration-and-cough-fivebreaths-3',
 'respiration-and-cough-fivebreaths-4',
 'respiration-and-cough-threequickbreaths-1',
 'respiration-and-cough-threequickbreaths-2',
 'story-recall',
 'word-color-stroop',
 'breath-sounds',
 

In [218]:
pk_train_c['participant_id'].unique()

<ArrowStringArray>
['110043', '039709', '844158', '024013', '193039', '525316', '891905',
 '403436', '234665', '237791', '982368', '579465', '852129', '267027',
 '686795', '672685', '401731', '505795', '424233', '943245', '881394',
 '090051', '524544', '684449', '371251', '260036', '124647', '420248',
 '394760', '192813', '478433', '024125', '629473', '120723', '075587',
 '108713', '145492', '962050', '748240', '174502', '141858', '427735',
 '312782', '695754', '113118', '958051', '748807', '224730', '959779',
 '013776', '316915', '320532', '833079', '265997', '641284', '968296']
Length: 56, dtype: str

## 4. Handle remaining NaN values
For rows that survived the column drop:
- **Numeric columns**: fill with median (robust to outliers) or leave as-is depending on downstream model.
- **Categorical columns**: fill with `"Unknown"` or most frequent value.
- **Audio features** (`mfcc`, `mel_spectrogram`): should have 0% NaN; flag any issues.

In [219]:
# ══════════════════════════════════════════════════════════════════
#  5. Remaining NaN report + imputation strategy
# ══════════════════════════════════════════════════════════════════

def nan_report(df, name):
    """Show remaining NaN per column after cleaning."""
    n_rows = len(df)
    nan_cols = []
    for col in df.columns:
        n_na = df[col].isna().sum()
        if n_na > 0:
            pct = 100 * n_na / n_rows
            nan_cols.append({"column": col, "NaN": n_na, "NaN%": round(pct, 1),
                             "dtype": str(df[col].dtype)})

    if nan_cols:
        nan_df = pd.DataFrame(nan_cols).sort_values("NaN%", ascending=False)
        print(f"\n{'═' * 70}")
        print(f"  {name} — {len(nan_cols)} columns still have NaN")
        print(f"{'═' * 70}")
        display(nan_df)
    else:
        print(f"\n  ✔  {name} — No NaN values remaining!")

    return pd.DataFrame(nan_cols) if nan_cols else pd.DataFrame()

train_nans = nan_report(pk_train_c, "pk_train_c")


══════════════════════════════════════════════════════════════════════
  pk_train_c — 43 columns still have NaN
══════════════════════════════════════════════════════════════════════


,column,NaN,NaN%,dtype
8,household_income_usa,1212,64.6,float64
19,marital_status___2,950,50.6,float64
29,self_reported_huntingtons,933,49.7,float64
9,housing_status,689,36.7,float64
20,race___5,491,26.2,float64
18,employ_status___7,377,20.1,float64
17,demographics_completed_by___1,374,19.9,float64
5,grandparent,373,19.9,float64
12,other_live_with,373,19.9,float64
13,parent,373,19.9,float64


In [220]:
# ══════════════════════════════════════════════════════════════════
#  5b. Impute remaining NaN values
# ══════════════════════════════════════════════════════════════════

def impute_nans(df, name):
    """
    - Numeric columns: fill NaN with column median
    - Object/string columns: fill NaN with 'Unknown'
    - Audio columns (mfcc, mel_spectrogram): leave as-is (should be 0% NaN)
    """
    df = df.copy()
    skip_cols = {"participant_id", "session_id", "task_name", "mfcc", "mel_spectrogram"}
    filled = {"numeric": 0, "categorical": 0}

    for col in df.columns:
        if col in skip_cols:
            continue
        n_na = df[col].isna().sum()
        if n_na == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            filled["numeric"] += 1
        else:
            df[col] = df[col].fillna("Unknown")
            filled["categorical"] += 1

    remaining_na = df.isna().sum().sum()
    print(f"  [{name}] Imputed: {filled['numeric']} numeric (median), "
          f"{filled['categorical']} categorical ('Unknown')")
    print(f"  [{name}] Remaining NaN: {remaining_na}")
    return df

pk_train_c = impute_nans(pk_train_c, "pk_train_c")

  [pk_train_c] Imputed: 43 numeric (median), 0 categorical ('Unknown')
  [pk_train_c] Remaining NaN: 0


## 6. Final summary & save cleaned tables

In [221]:
# ══════════════════════════════════════════════════════════════════
#  6. Final summary & save
# ══════════════════════════════════════════════════════════════════

def final_summary(df, name):
    n_pids = df["participant_id"].nunique()
    n_sess = df["session_id"].nunique()
    n_tasks = df["task_name"].nunique() if "task_name" in df.columns else 0
    n_na_total = df.isna().sum().sum()

    print(f"{'═' * 70}")
    print(f"  {name}")
    print(f"{'─' * 70}")
    print(f"  Shape:        {df.shape}")
    print(f"  Participants: {n_pids}")
    print(f"  Sessions:     {n_sess}")
    print(f"  Tasks:        {n_tasks}")
    print(f"  Total NaN:    {n_na_total}")
    print(f"  Dtypes:       {df.dtypes.value_counts().to_dict()}")
    print(f"{'═' * 70}\n")

final_summary(pk_train_c, "pk_train_c (non-longitudinal, cleaned)")

# ── Save cleaned table ───────────────────────────────────────────
SAVE_DIR = Path("/home/srl/B2AI/LLM/data")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

clean_path = SAVE_DIR / "cleaned_non_longitudinal_parkinson.parquet"
pk_train_c.to_parquet(clean_path, index=False)

print(f"✅  Saved cleaned table to {SAVE_DIR}/")
print(f"  {clean_path.name:55s}  {pk_train_c.shape}")

══════════════════════════════════════════════════════════════════════
  pk_train_c (non-longitudinal, cleaned)
──────────────────────────────────────────────────────────────────────
  Shape:        (1876, 73)
  Participants: 56
  Sessions:     56
  Tasks:        120
  Total NaN:    0
  Dtypes:       {dtype('float64'): 46, dtype('int64'): 18, Int8Dtype(): 4, <StringDtype(na_value=nan)>: 3, dtype('O'): 2}
══════════════════════════════════════════════════════════════════════

✅  Saved cleaned table to /home/srl/B2AI/LLM/data/
  cleaned_non_longitudinal_parkinson.parquet               (1876, 73)


## 7. Clean the test (longitudinal) dataset
Apply the **same** cleaning pipeline as the train set, then align columns so the **feature** schema matches train.

**Key raw-data differences from train:**
- Column `parkinsons_disease__diagnosis_parkinsons_ds` → rename to `diagnosis_parkinsons_ds`
- Only `vhi10__vhi_10_calc_score` is dropped from raw test; **`voice_perception__voice_quality_perception` is kept** through cleaning, then temporarily removed during column alignment (7c) and **merged back** so the saved test parquet has **one extra column** vs train: the voice-quality score (plus `session_id` is already in both).
- Some columns may be >60 % NaN in test but not in train → added during alignment, filled with train median

In [222]:
# ══════════════════════════════════════════════════════════════════
#  7a. Load test (longitudinal) data + harmonise column names
# ══════════════════════════════════════════════════════════════════

pk_test = pd.read_parquet(DATA_DIR / "raw_data_longitudinal_parkinson.parquet")
print(f"Loaded test (longitudinal):  {pk_test.shape}")

# ── Rename to match train naming ──────────────────────────────────
pk_test = pk_test.rename(columns={
    "parkinsons_disease__diagnosis_parkinsons_ds": "diagnosis_parkinsons_ds",
})

# ── Drop columns that only exist in the test raw data ─────────────
EXTRA_TEST_COLS = ["vhi10__vhi_10_calc_score"]
pk_test = pk_test.drop(columns=[c for c in EXTRA_TEST_COLS if c in pk_test.columns])
print(f"After rename + drop extras:  {pk_test.shape}")

Loaded test (longitudinal):  (700, 137)
After rename + drop extras:  (700, 136)


In [223]:
# ── Quick inspect ──────────────────────────────────────────────────
_ = inspect_table(pk_test, "pk_test (longitudinal, raw)")


════════════════════════════════════════════════════════════════════════════════
  pk_test (longitudinal, raw)   shape: (700, 136)
════════════════════════════════════════════════════════════════════════════════


,column,dtype,NaN,NaN%,unique,sample
0,participant_id,str,0,0.0,6,225994
1,session_id,str,0,0.0,20,45b76d73
2,voice_perception__voice_quality_perception,float64,0,0.0,7,7.0
3,diagnosis_parkinsons_ds,float64,0,0.0,4,1.0
4,task_name,str,0,0.0,72,breath-sounds
...,...,...,...,...,...,...
131,eligible_studies___4,str,558,79.7,1,1.0
132,eligible_studies___age_10_plus,str,700,100.0,0,—
133,eligible_studies___age_2_4,str,700,100.0,0,—
134,eligible_studies___age_4_6,str,700,100.0,0,—



  ⚠  63 columns with > 60% NaN:
    • colorblind                                                93.6%
    • household_income_usa                                     100.0%
    • live_by_yourself                                          93.6%
    • roommate                                                  93.6%
    • sex_assigned_at_birth                                     93.6%
    • tone_deaf                                                 93.6%
    • transportation                                            93.6%
    • demographics_completed_by___2                            100.0%
    • demographics_completed_by___3                            100.0%
    • employ_status___1                                        100.0%
    • employ_status___10                                       100.0%
    • employ_status___11                                       100.0%
    • employ_status___12                                       100.0%
    • employ_status___13                                 

In [224]:
# ══════════════════════════════════════════════════════════════════
#  7b. Apply the SAME cleaning pipeline as train
# ══════════════════════════════════════════════════════════════════

# Step 1 — Drop columns with > 60% NaN
pk_test_c = drop_high_nan(pk_test, "pk_test (longitudinal)")

# Step 2 — Map ordinal / boolean / checked / one-hot + coerce types
pk_test_c = clean_types(pk_test_c, "pk_test_c")

# Step 3 — NaN report before imputation
test_nans = nan_report(pk_test_c, "pk_test_c")

# Step 4 — Impute remaining NaN (median for numeric, 'Unknown' for categorical)
pk_test_c = impute_nans(pk_test_c, "pk_test_c")

══════════════════════════════════════════════════════════════════════
  pk_test (longitudinal)
  Before: 136 columns
  Dropping 63 columns with > 60% NaN:
  After:  73 columns
══════════════════════════════════════════════════════════════════════

  [pk_test_c] Dropped: ['demographics_duration', 'ef_primary_language', 'ef_select_language']
  [pk_test_c] Ordinal  edu_level                                 (0 unmapped → NaN)
  [pk_test_c] Ordinal  housing_status                            (0 unmapped → NaN)
  [pk_test_c] One-hot  country                                   → ['country_Canada']
  [pk_test_c] One-hot  sexual_orientation                        → ['sexual_orientation_Heterosexual (straight)']

  [pk_test_c] SUMMARY:
    Ordinal mapped:   2 columns
    Boolean mapped:   18 columns
    Checked mapped:   31 self_reported_* columns
    One-hot encoded:  2 columns
    Numeric coerced:  3 columns
    Flag coerced:     6 (___) columns
    Final shape:      (700, 70)

════════════════

,column,NaN,NaN%,dtype
4,self_reported_glottic_insufficiency,414,59.1,float64
1,primary_transportation,324,46.3,str
3,ef_fluent_languages___3,309,44.1,float64
5,self_reported_laryngitis,278,39.7,float64
2,demographics_completed_by___1,45,6.4,float64
0,housing_status,36,5.1,float64


  [pk_test_c] Imputed: 5 numeric (median), 1 categorical ('Unknown')
  [pk_test_c] Remaining NaN: 0


In [225]:
# ══════════════════════════════════════════════════════════════════
#  7c. Align test columns to match cleaned train EXACTLY
#      - Add columns that are in train but missing in test
#        (filled with train median for numeric, 0 for binary/flag)
#      - Drop columns that are in test but NOT in train
#      - Reorder to match train column order
#      - Preserve test-only target: voice_perception__voice_quality_perception
#        (snapshot + re-merge after alignment; not in train schema)
# ══════════════════════════════════════════════════════════════════

VOICE_COL = "voice_perception__voice_quality_perception"
test_voice_session = None
if VOICE_COL in pk_test_c.columns:
    test_voice_session = (
        pk_test_c.groupby(["participant_id", "session_id"], as_index=False)[VOICE_COL]
        .first()
    )
    pk_test_c = pk_test_c.drop(columns=[VOICE_COL])

train_cols = list(pk_train_c.columns)

missing_in_test = [c for c in train_cols if c not in pk_test_c.columns]
extra_in_test   = [c for c in pk_test_c.columns if c not in train_cols]

print(f"Columns in train but MISSING in test ({len(missing_in_test)}):")
for c in missing_in_test:
    print(f"  • {c}")

print(f"\nColumns in test but NOT in train ({len(extra_in_test)}):")
for c in extra_in_test:
    print(f"  • {c}")

# ── Add missing columns using train statistics ────────────────────
for col in missing_in_test:
    if pd.api.types.is_numeric_dtype(pk_train_c[col]):
        fill_val = pk_train_c[col].median()
        pk_test_c[col] = fill_val
        print(f"  Added '{col}' → filled with train median ({fill_val})")
    else:
        pk_test_c[col] = 0
        print(f"  Added '{col}' → filled with 0")

# ── Drop extra columns ────────────────────────────────────────────
if extra_in_test:
    pk_test_c = pk_test_c.drop(columns=extra_in_test)
    print(f"\n  Dropped {len(extra_in_test)} extra columns: {extra_in_test}")

# ── Reorder to match train ────────────────────────────────────────
pk_test_c = pk_test_c[train_cols]

# ── Verify ─────────────────────────────────────────────────────────
assert list(pk_test_c.columns) == train_cols, "❌ Column mismatch!"
print(f"\n✅  Test columns now match train — {len(train_cols)} columns")

# ── Re-attach longitudinal voice-quality target (test only) ───────
if test_voice_session is not None:
    pk_test_c = pk_test_c.merge(
        test_voice_session, on=["participant_id", "session_id"], how="left"
    )
    print(f"  Re-attached {VOICE_COL} ({pk_test_c[VOICE_COL].notna().sum()} non-null rows)")

Columns in train but MISSING in test (6):
  • household_income_usa
  • marital_status___2
  • age
  • self_reported_huntingtons
  • country_USA
  • sexual_orientation_Bisexual

Columns in test but NOT in train (2):
  • primary_transportation
  • ef_fluent_languages___3
  Added 'household_income_usa' → filled with train median (4.0)
  Added 'marital_status___2' → filled with train median (1.0)
  Added 'age' → filled with train median (72.0)
  Added 'self_reported_huntingtons' → filled with train median (0.0)
  Added 'country_USA' → filled with train median (1.0)
  Added 'sexual_orientation_Bisexual' → filled with train median (0.0)

  Dropped 2 extra columns: ['primary_transportation', 'ef_fluent_languages___3']

✅  Test columns now match train — 73 columns
  Re-attached voice_perception__voice_quality_perception (700 non-null rows)


In [226]:
# ══════════════════════════════════════════════════════════════════
#  7d. Final summary & save cleaned test table
# ══════════════════════════════════════════════════════════════════

final_summary(pk_test_c, "pk_test_c (longitudinal, cleaned)")

# ── Save ──────────────────────────────────────────────────────────
test_clean_path = SAVE_DIR / "cleaned_longitudinal_parkinson.parquet"
pk_test_c.to_parquet(test_clean_path, index=False)

print(f"✅  Saved cleaned test table to {SAVE_DIR}/")
print(f"  {test_clean_path.name:55s}  {pk_test_c.shape}")

# ── Side-by-side comparison ───────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"  TRAIN vs TEST — column check")
print(f"{'─' * 70}")
print(f"  Train columns: {len(pk_train_c.columns)}")
print(f"  Test  columns: {len(pk_test_c.columns)}")
print(f"  Match:         {list(pk_train_c.columns) == list(pk_test_c.columns)}")
print(f"  Train shape:   {pk_train_c.shape}")
print(f"  Test  shape:   {pk_test_c.shape}")
print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
  pk_test_c (longitudinal, cleaned)
──────────────────────────────────────────────────────────────────────
  Shape:        (700, 74)
  Participants: 6
  Sessions:     20
  Tasks:        72
  Total NaN:    0
  Dtypes:       {dtype('float64'): 35, dtype('int64'): 32, <StringDtype(na_value=nan)>: 3, dtype('O'): 2, Int8Dtype(): 2}
══════════════════════════════════════════════════════════════════════

✅  Saved cleaned test table to /home/srl/B2AI/LLM/data/
  cleaned_longitudinal_parkinson.parquet                   (700, 74)

══════════════════════════════════════════════════════════════════════
  TRAIN vs TEST — column check
──────────────────────────────────────────────────────────────────────
  Train columns: 73
  Test  columns: 74
  Match:         False
  Train shape:   (1876, 73)
  Test  shape:   (700, 74)
══════════════════════════════════════════════════════════════════════


In [227]:
pk_train_c

,participant_id,session_id,task_name,mfcc,n_frames_mfcc,mel_spectrogram,n_frames_mel,diagnosis_parkinsons_ds,children,cognition,...,self_reported_rrp,self_reported_schizophrenia,self_reported_soc_anx_dis,self_reported_spas_dys,self_reported_unilateral_vocal_fold_paralysis,eligible_studies___2,country_Canada,country_USA,sexual_orientation_Bisexual,sexual_orientation_Heterosexual (straight)
0,110043,2cd1e0d6,cinderella-story,"[[-158.8, -161.9, -200.4, -218.1, -276.8, -183...",3278,"[[0.00443, 0.02866, 0.001359, 0.00585, 0.00919...",3278,1.0,0.0,0.0,...,0,0.0,0.0,0,0,1.0,0,1,0,1
1,110043,2cd1e0d6,diadochokinesis-buttercup,"[[-313.8, -311.2, -315.2, -319.8, -308.2, -307...",558,"[[0.0598, 0.00697, 0.0007863, 0.01677, 0.0082,...",558,1.0,0.0,0.0,...,0,0.0,0.0,0,0,1.0,0,1,0,1
2,110043,2cd1e0d6,diadochokinesis-ka,"[[-328.2, -309.2, -306.0, -232.8, -224.8, -273...",711,"[[0.013985, 0.01135, 0.001665, 0.01376, 0.063,...",711,1.0,0.0,0.0,...,0,0.0,0.0,0,0,1.0,0,1,0,1
3,110043,2cd1e0d6,diadochokinesis-pa,"[[-241.6, -308.5, -307.8, -308.0, -317.2, -310...",513,"[[0.3906, 0.4207, 0.9053, 0.348, 0.513, 0.084,...",513,1.0,0.0,0.0,...,0,0.0,0.0,0,0,1.0,0,1,0,1
4,110043,2cd1e0d6,diadochokinesis-pataka,"[[-330.0, -315.5, -310.8, -307.0, -306.5, -311...",585,"[[0.0001646, 0.005596, 0.00842, 0.1225, 0.0702...",585,1.0,0.0,0.0,...,0,0.0,0.0,0,0,1.0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1871,968296,cd772e40,respiration-and-cough-fivebreaths-4,"[[-159.0, -167.5, -162.1, -155.9, -160.5, -170...",728,"[[2.682, 0.0318, 0.3564, 2.46, 0.4873, 1.049, ...",728,3.0,0.0,1.0,...,0,0.0,0.0,0,0,1.0,1,0,0,1
1872,968296,cd772e40,respiration-and-cough-threequickbreaths-1,"[[-47.72, -30.2, -16.39, -19.72, -22.12, -42.6...",286,"[[35.22, 36.1, 25.02, 20.0, 27.88, 5.74, 7.59,...",286,3.0,0.0,1.0,...,0,0.0,0.0,0,0,1.0,1,0,0,1
1873,968296,cd772e40,respiration-and-cough-threequickbreaths-2,"[[-189.2, -195.1, -204.9, -218.5, -225.8, -204...",251,"[[3.41, 7.973, 2.88, 3.127, 2.793, 8.55, 0.051...",251,3.0,0.0,1.0,...,0,0.0,0.0,0,0,1.0,1,0,0,1
1874,968296,cd772e40,story-recall,"[[-331.8, -346.2, -359.8, -363.2, -367.2, -363...",1949,"[[0.05902, 0.2046, 0.1501, 0.02682, 0.02621, 0...",1949,3.0,0.0,1.0,...,0,0.0,0.0,0,0,1.0,1,0,0,1


## 8. Audio Feature Processing

Both **MFCC** and **mel spectrogram** are stored as variable-length 2-D arrays
`(n_coeff, n_frames)` — 60 coefficients/bins, variable time.

### MFCC pipeline
1. **Per-recording z-normalisation** — subtract mean, divide by std across the time axis
   for each coefficient → zero-mean, unit-variance per recording.
2. **Delta (Δ)** — 1st-order finite difference along time for each coefficient.
3. **Delta-delta (ΔΔ)** — 2nd-order finite difference (delta of delta).
4. Stack originals + Δ + ΔΔ → `(180, T)` per recording.

### Mel spectrogram pipeline
1. **Per-recording z-normalisation** — same z-score as MFCC.
2. **Fixed-size padding / truncation** — pad short recordings (zero) or truncate
   long recordings to a fixed number of frames (T_FIXED = 512, close to the median).

In [228]:
# ══════════════════════════════════════════════════════════════════
#  8a. Helper functions for audio processing
# ══════════════════════════════════════════════════════════════════

T_FIXED = 512  # fixed number of frames for mel spectrogram (≈ median)

def to_2d(arr):
    """Convert a stored array (list-of-lists or object-array) to float32 2D ndarray."""
    return np.stack(arr).astype(np.float32)   # (n_coeff, n_frames)


def znorm_per_recording(mat):
    """Z-normalise each coefficient across time:  (x - μ) / (σ + ε)."""
    mu  = mat.mean(axis=1, keepdims=True)
    std = mat.std(axis=1, keepdims=True) + 1e-8
    return (mat - mu) / std


def compute_deltas(mat):
    """
    Compute Δ (1st-order) and ΔΔ (2nd-order) along the time axis.
    Returns stacked array: (3·n_coeff, n_frames).
    Uses np.gradient for smooth finite differences (same length as input).
    """
    delta  = np.gradient(mat, axis=1).astype(np.float32)    # (n_coeff, T)
    ddelta = np.gradient(delta, axis=1).astype(np.float32)  # (n_coeff, T)
    return np.concatenate([mat, delta, ddelta], axis=0)      # (3·n_coeff, T)


def pad_or_truncate(mat, target_len):
    """Pad (with zeros) or truncate along the time axis (axis=1)."""
    T = mat.shape[1]
    if T >= target_len:
        return mat[:, :target_len]
    else:
        pad_width = target_len - T
        return np.pad(mat, ((0, 0), (0, pad_width)), mode="constant", constant_values=0)


print("✔  Audio processing helpers defined")
print(f"   T_FIXED = {T_FIXED} frames for mel spectrogram")

✔  Audio processing helpers defined
   T_FIXED = 512 frames for mel spectrogram


In [229]:
# ══════════════════════════════════════════════════════════════════
#  8b. Process MFCCs — z-norm + delta + delta-delta
#      Applied to BOTH train and test
# ══════════════════════════════════════════════════════════════════

def process_mfcc_column(df, name):
    """
    For each row:
      1. Convert stored mfcc to 2D float32 → (60, T)
      2. Z-normalise per recording
      3. Compute delta + delta-delta → (180, T)
    Stores result back in the 'mfcc' column as a numpy array.
    """
    processed = []
    n = len(df)
    for i, raw in enumerate(df["mfcc"]):
        mat = to_2d(raw)                    # (60, T)
        mat = znorm_per_recording(mat)      # (60, T) z-normed
        mat = compute_deltas(mat)           # (180, T)
        processed.append(mat)
        if (i + 1) % 500 == 0 or i == n - 1:
            print(f"  [{name}] MFCC processed {i+1}/{n}", end="\r")

    df = df.copy()
    df["mfcc"] = processed
    print(f"\n  [{name}] MFCC done — shape per recording: "
          f"({processed[0].shape[0]}, variable T)")
    return df

pk_train_c = process_mfcc_column(pk_train_c, "train")
pk_test_c  = process_mfcc_column(pk_test_c,  "test")

# Quick sanity check
sample = pk_train_c["mfcc"].iloc[0]
print(f"\n  Sanity: train row 0 mfcc shape = {sample.shape}, "
      f"mean ≈ {sample.mean():.4f}, std ≈ {sample.std():.4f}")

  [train] MFCC processed 1876/1876
  [train] MFCC done — shape per recording: (180, variable T)
  [test] MFCC processed 700/700
  [test] MFCC done — shape per recording: (180, variable T)

  Sanity: train row 0 mfcc shape = (180, 3278), mean ≈ 0.0000, std ≈ 0.7189


In [230]:
# ══════════════════════════════════════════════════════════════════
#  8c. Process Mel Spectrograms — z-norm + fixed-size
#      Applied to BOTH train and test
# ══════════════════════════════════════════════════════════════════

def process_mel_column(df, name, target_len=T_FIXED):
    """
    For each row:
      1. Convert stored mel_spectrogram to 2D float32 → (60, T)
      2. Z-normalise per recording
      3. Pad / truncate to (60, target_len)
    Stores result back in the 'mel_spectrogram' column.
    """
    processed = []
    n = len(df)
    for i, raw in enumerate(df["mel_spectrogram"]):
        mat = to_2d(raw)                         # (60, T)
        mat = znorm_per_recording(mat)           # (60, T) z-normed
        mat = pad_or_truncate(mat, target_len)   # (60, target_len)
        processed.append(mat)
        if (i + 1) % 500 == 0 or i == n - 1:
            print(f"  [{name}] Mel processed {i+1}/{n}", end="\r")

    df = df.copy()
    df["mel_spectrogram"] = processed
    print(f"\n  [{name}] Mel done — fixed shape per recording: {processed[0].shape}")
    return df

pk_train_c = process_mel_column(pk_train_c, "train")
pk_test_c  = process_mel_column(pk_test_c,  "test")

# Quick sanity check
sample = pk_train_c["mel_spectrogram"].iloc[0]
print(f"\n  Sanity: train row 0 mel shape = {sample.shape}, "
      f"mean ≈ {sample.mean():.4f}, std ≈ {sample.std():.4f}")

  [train] Mel processed 1876/1876
  [train] Mel done — fixed shape per recording: (60, 512)
  [test] Mel processed 700/700
  [test] Mel done — fixed shape per recording: (60, 512)

  Sanity: train row 0 mel shape = (60, 512), mean ≈ -0.0161, std ≈ 0.8524


## 10. Save final processed datasets

In [231]:
# ══════════════════════════════════════════════════════════════════
#  10. Save final processed datasets (overwrite earlier cleaned files)
# ══════════════════════════════════════════════════════════════════
#
#  Parquet (Arrow) cannot store 2-D numpy arrays directly.
#  → Convert mfcc / mel_spectrogram columns to list-of-lists before saving.
#  When loading back: df["mfcc"].apply(lambda x: np.array(x, dtype=np.float32))

ARRAY_COLS = ["mfcc", "mel_spectrogram"]

def _arrays_to_lists(df):
    """Convert 2-D numpy array columns to nested Python lists for Arrow."""
    df = df.copy()
    for col in ARRAY_COLS:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
    return df

# ── Train ─────────────────────────────────────────────────────────
train_final_path = SAVE_DIR / "cleaned_non_longitudinal_parkinson.parquet"
_arrays_to_lists(pk_train_c).to_parquet(train_final_path, index=False)

# ── Test ──────────────────────────────────────────────────────────
test_final_path = SAVE_DIR / "cleaned_longitudinal_parkinson.parquet"
_arrays_to_lists(pk_test_c).to_parquet(test_final_path, index=False)

# ── Final summary ─────────────────────────────────────────────────
print(f"{'═' * 70}")
print(f"  FINAL PROCESSED DATASETS")
print(f"{'═' * 70}")

print(f"\n  TRAIN (non-longitudinal)")
print(f"{'─' * 70}")
print(f"  Shape:          {pk_train_c.shape}")
print(f"  Participants:   {pk_train_c['participant_id'].nunique()}")
print(f"  MFCC shape:     ({pk_train_c['mfcc'].iloc[0].shape[0]}, variable T)  "
      f"[z-normed + Δ + ΔΔ]")
print(f"  Mel shape:      {pk_train_c['mel_spectrogram'].iloc[0].shape}  "
      f"[z-normed + fixed-size]")
print(f"  Saved to:       {train_final_path}")

print(f"\n  TEST (longitudinal)")
print(f"{'─' * 70}")
print(f"  Shape:          {pk_test_c.shape}")
print(f"  Participants:   {pk_test_c['participant_id'].nunique()}")
print(f"  Sessions:       {pk_test_c['session_id'].nunique()}")
print(f"  MFCC shape:     ({pk_test_c['mfcc'].iloc[0].shape[0]}, variable T)  "
      f"[z-normed + Δ + ΔΔ]")
print(f"  Mel shape:      {pk_test_c['mel_spectrogram'].iloc[0].shape}  "
      f"[z-normed + fixed-size]")
print(f"  Raw targets:    diagnosis_parkinsons_ds, voice_perception__voice_quality_perception")
print(f"  Saved to:       {test_final_path}")
print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
  FINAL PROCESSED DATASETS
══════════════════════════════════════════════════════════════════════

  TRAIN (non-longitudinal)
──────────────────────────────────────────────────────────────────────
  Shape:          (1876, 73)
  Participants:   56
  MFCC shape:     (180, variable T)  [z-normed + Δ + ΔΔ]
  Mel shape:      (60, 512)  [z-normed + fixed-size]
  Saved to:       /home/srl/B2AI/LLM/data/cleaned_non_longitudinal_parkinson.parquet

  TEST (longitudinal)
──────────────────────────────────────────────────────────────────────
  Shape:          (700, 74)
  Participants:   6
  Sessions:       20
  MFCC shape:     (180, variable T)  [z-normed + Δ + ΔΔ]
  Mel shape:      (60, 512)  [z-normed + fixed-size]
  Raw targets:    diagnosis_parkinsons_ds, voice_perception__voice_quality_perception
  Saved to:       /home/srl/B2AI/LLM/data/cleaned_longitudinal_parkinson.parquet
═══════════════════════════════════════════════

In [232]:
# ══════════════════════════════════════════════════════════════════
#  11. Column comparison — Train vs Test
# ══════════════════════════════════════════════════════════════════

train_cols = set(pk_train_c.columns)
test_cols  = set(pk_test_c.columns)

only_in_train = sorted(train_cols - test_cols)
only_in_test  = sorted(test_cols - train_cols)
shared        = sorted(train_cols & test_cols)

print(f"{'═' * 70}")
print(f"  COLUMN COMPARISON — Train ({len(train_cols)} cols) vs Test ({len(test_cols)} cols)")
print(f"{'═' * 70}")

print(f"\n  Shared columns: {len(shared)}")

if only_in_train:
    print(f"\n  Columns ONLY in Train ({len(only_in_train)}):")
    for c in only_in_train:
        print(f"    • {c}")
else:
    print(f"\n  ✔ No columns only in train")

if only_in_test:
    print(f"\n  Columns ONLY in Test ({len(only_in_test)}):")
    for c in only_in_test:
        print(f"    • {c}")
else:
    print(f"\n  ✔ No columns only in test")

if not only_in_train and not only_in_test:
    print(f"\n  ✅ Train and test have identical columns!")

print(f"{'═' * 70}")

══════════════════════════════════════════════════════════════════════
  COLUMN COMPARISON — Train (73 cols) vs Test (74 cols)
══════════════════════════════════════════════════════════════════════

  Shared columns: 73

  ✔ No columns only in train

  Columns ONLY in Test (1):
    • voice_perception__voice_quality_perception
══════════════════════════════════════════════════════════════════════
